In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/cleaned/final_analysis_data.csv")

In [ ]:
df = df.sort_values(['country', 'year'])

df = df[df['co2_per_capita_3yr_change'].notna()]

In [21]:
Y = ['co2_per_capita_3yr_change']
treatment = ['post_carbon_price', 'years_relative_to_treatment']
controls = ['log_gdp', 'log_population', 'log_energy_per_capita', 'industry_share_gdp',
            'fossil_pct', 'natural_resource_rents_per_gdp', 'implementation_capacity_z', 'political_stability_z', 'democratic_legitimacy_z']
id_cols = ['country', 'year']
required_cols = id_cols + Y + treatment + controls
df_did = df[required_cols].copy()

In [23]:
df_did = df_did.dropna(
    subset=Y + ["post_carbon_price"] + controls
)
df_did = df_did.sort_values(["country", "year"])

In [24]:
df_did

,country,year,co2_per_capita_3yr_change,post_carbon_price,years_relative_to_treatment,log_gdp,log_population,log_energy_per_capita,industry_share_gdp,fossil_pct,natural_resource_rents_per_gdp,implementation_capacity_z,political_stability_z,democratic_legitimacy_z
6,Afghanistan,2002,0.014,0,NaN,23.657382,16.877879,5.351204,23.810127,62.20,1.276149,-1.663913,-0.913638,-0.246275
7,Afghanistan,2003,0.016,0,NaN,23.771322,16.939331,5.437940,22.710864,63.30,0.731313,-1.500448,-1.528341,-0.182162
8,Afghanistan,2004,0.055,0,NaN,23.829312,16.975089,5.319252,26.226790,55.80,0.458920,-1.513576,-1.683727,-0.343972
9,Afghanistan,2005,0.084,0,NaN,23.957924,17.010281,5.529703,26.812099,66.10,0.393193,-1.551769,-1.241759,0.110182
10,Afghanistan,2006,0.148,0,NaN,24.080316,17.051208,5.718412,28.210768,68.10,0.504174,-1.669319,-1.383848,0.332902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3887,Zimbabwe,2015,-0.090,0,NaN,23.911524,16.482670,8.258661,22.358392,50.78,4.606185,-1.244063,0.962376,0.327073
3888,Zimbabwe,2016,-0.053,0,NaN,23.916292,16.496552,8.079519,22.115059,47.58,4.495414,-1.256476,0.985411,0.307862
3889,Zimbabwe,2017,-0.116,0,NaN,23.964921,16.510981,8.028785,32.015626,44.75,6.095448,-1.258894,0.811028,0.251626
3890,Zimbabwe,2018,-0.099,0,NaN,24.011155,16.525855,8.143804,31.037898,47.19,3.378189,-1.215470,0.735085,0.282538


In [26]:
import statsmodels.formula.api as smf

formula = """
co2_per_capita_3yr_change ~ post_carbon_price
    + log_gdp
    + log_population
    + log_energy_per_capita
    + industry_share_gdp
    + fossil_pct
    + natural_resource_rents_per_gdp
    + implementation_capacity_z
    + political_stability_z
    + democratic_legitimacy_z
    + C(country) + C(year)
"""

did_analysis = smf.ols(formula=formula, data=df_did).fit(
    cov_type = 'cluster',
    cov_kwds = {"groups": df_did['country']}
)

print(did_analysis.summary())

                                OLS Regression Results                               
Dep. Variable:     co2_per_capita_3yr_change   R-squared:                       0.217
Model:                                   OLS   Adj. R-squared:                  0.175
Method:                        Least Squares   F-statistic:                     5.366
Date:                       Sat, 20 Dec 2025   Prob (F-statistic):           1.84e-13
Time:                               09:43:28   Log-Likelihood:                -5183.6
No. Observations:                       3830   AIC:                         1.076e+04
Df Residuals:                           3635   BIC:                         1.198e+04
Df Model:                                194                                         
Covariance Type:                     cluster                                         
                                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

C:\Users\willn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1888: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 194, but rank is 33
  warnings.warn('covariance of constraints does not have full '


In [28]:
import statsmodels.formula.api as smf

formula_implementation = """
co2_per_capita_3yr_change ~ post_carbon_price*implementation_capacity_z
    + log_gdp
    + log_population
    + log_energy_per_capita
    + industry_share_gdp
    + fossil_pct
    + natural_resource_rents_per_gdp
    + implementation_capacity_z
    + political_stability_z
    + democratic_legitimacy_z
    + C(country) + C(year)
"""

did_analysis_implementation = smf.ols(formula=formula_implementation, data=df_did).fit(
    cov_type = 'cluster',
    cov_kwds = {"groups": df_did['country']}
)

print(did_analysis_implementation.summary())

                                OLS Regression Results                               
Dep. Variable:     co2_per_capita_3yr_change   R-squared:                       0.220
Model:                                   OLS   Adj. R-squared:                  0.178
Method:                        Least Squares   F-statistic:                     5.133
Date:                       Sat, 20 Dec 2025   Prob (F-statistic):           5.42e-13
Time:                               18:30:15   Log-Likelihood:                -5177.2
No. Observations:                       3830   AIC:                         1.075e+04
Df Residuals:                           3634   BIC:                         1.197e+04
Df Model:                                195                                         
Covariance Type:                     cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
----------------------------

C:\Users\willn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1888: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 195, but rank is 34
  warnings.warn('covariance of constraints does not have full '


In [32]:
formula_stability = """
co2_per_capita_3yr_change ~ post_carbon_price*political_stability_z
    + log_gdp
    + log_population
    + log_energy_per_capita
    + industry_share_gdp
    + fossil_pct
    + natural_resource_rents_per_gdp
    + implementation_capacity_z
    + democratic_legitimacy_z
    + C(country) + C(year)
"""

did_analysis_stability = smf.ols(formula=formula_stability, data=df_did).fit(
    cov_type = 'cluster',
    cov_kwds = {"groups": df_did['country']}
)

print(did_analysis_stability.summary())

                                OLS Regression Results                               
Dep. Variable:     co2_per_capita_3yr_change   R-squared:                       0.217
Model:                                   OLS   Adj. R-squared:                  0.175
Method:                        Least Squares   F-statistic:                     5.195
Date:                       Sat, 20 Dec 2025   Prob (F-statistic):           3.58e-13
Time:                               18:43:43   Log-Likelihood:                -5183.3
No. Observations:                       3830   AIC:                         1.076e+04
Df Residuals:                           3634   BIC:                         1.198e+04
Df Model:                                195                                         
Covariance Type:                     cluster                                         
                                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

C:\Users\willn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1888: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 195, but rank is 34
  warnings.warn('covariance of constraints does not have full '


In [33]:
formula_democratic = """
co2_per_capita_3yr_change ~ post_carbon_price*democratic_legitimacy_z
    + log_gdp
    + log_population
    + log_energy_per_capita
    + industry_share_gdp
    + fossil_pct
    + natural_resource_rents_per_gdp
    + implementation_capacity_z
    + political_stability_z
    + C(country) + C(year)
"""

did_analysis_democratic = smf.ols(formula=formula_democratic, data=df_did).fit(
    cov_type = 'cluster',
    cov_kwds = {"groups": df_did['country']}
)

print(did_analysis_democratic.summary())

                                OLS Regression Results                               
Dep. Variable:     co2_per_capita_3yr_change   R-squared:                       0.218
Model:                                   OLS   Adj. R-squared:                  0.176
Method:                        Least Squares   F-statistic:                     5.188
Date:                       Sat, 20 Dec 2025   Prob (F-statistic):           3.75e-13
Time:                               18:44:04   Log-Likelihood:                -5182.2
No. Observations:                       3830   AIC:                         1.076e+04
Df Residuals:                           3634   BIC:                         1.198e+04
Df Model:                                195                                         
Covariance Type:                     cluster                                         
                                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

C:\Users\willn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1888: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 195, but rank is 34
  warnings.warn('covariance of constraints does not have full '


In [34]:
formula_fossil_pct = """
co2_per_capita_3yr_change ~ post_carbon_price*fossil_pct
    + log_gdp
    + log_population
    + log_energy_per_capita
    + industry_share_gdp
    + political_stability_z
    + natural_resource_rents_per_gdp
    + implementation_capacity_z
    + democratic_legitimacy_z
    + C(country) + C(year)
"""

did_analysis_fossil_pct = smf.ols(formula=formula_fossil_pct, data=df_did).fit(
    cov_type = 'cluster',
    cov_kwds = {"groups": df_did['country']}
)

print(did_analysis_fossil_pct.summary())

                                OLS Regression Results                               
Dep. Variable:     co2_per_capita_3yr_change   R-squared:                       0.219
Model:                                   OLS   Adj. R-squared:                  0.177
Method:                        Least Squares   F-statistic:                     5.524
Date:                       Sat, 20 Dec 2025   Prob (F-statistic):           4.06e-14
Time:                               18:51:35   Log-Likelihood:                -5178.9
No. Observations:                       3830   AIC:                         1.075e+04
Df Residuals:                           3634   BIC:                         1.198e+04
Df Model:                                195                                         
Covariance Type:                     cluster                                         
                                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

C:\Users\willn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1888: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 195, but rank is 34
  warnings.warn('covariance of constraints does not have full '


In [35]:
formula_gdp = """
co2_per_capita_3yr_change ~ post_carbon_price*log_gdp 
    + fossil_pct
    + log_population
    + log_energy_per_capita
    + industry_share_gdp
    + political_stability_z
    + natural_resource_rents_per_gdp
    + implementation_capacity_z
    + democratic_legitimacy_z
    + C(country) + C(year)
"""

did_analysis_gdp = smf.ols(formula=formula_gdp, data=df_did).fit(
    cov_type = 'cluster',
    cov_kwds = {"groups": df_did['country']}
)

print(did_analysis_gdp.summary())

                                OLS Regression Results                               
Dep. Variable:     co2_per_capita_3yr_change   R-squared:                       0.219
Model:                                   OLS   Adj. R-squared:                  0.177
Method:                        Least Squares   F-statistic:                     5.353
Date:                       Sat, 20 Dec 2025   Prob (F-statistic):           1.25e-13
Time:                               19:40:25   Log-Likelihood:                -5180.2
No. Observations:                       3830   AIC:                         1.075e+04
Df Residuals:                           3634   BIC:                         1.198e+04
Df Model:                                195                                         
Covariance Type:                     cluster                                         
                                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

C:\Users\willn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1888: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 195, but rank is 34
  warnings.warn('covariance of constraints does not have full '


In [41]:
df_es = df_did.copy()

# 1. Event time
df_es["rel_year"] = df_es["years_relative_to_treatment"]

# 2. Clip to window [-K, K]
K = 8  # you can change to 10 if you prefer
df_es["rel_year_clipped"] = df_es["rel_year"].clip(lower=-K, upper=K)

# 3. Create dummy cols rel_-K, ..., rel_K
for k in range(-K, K+1):
    col = f"rel_{k}"
    df_es[col] = (df_es["rel_year_clipped"] == k).astype(int)

In [42]:
baseline = -1
event_dummies = [f"rel_{k}" for k in range(-K, K+1) if k != baseline]


In [44]:
cols_for_na_check = (
    ["co2_per_capita_3yr_change",
     "log_gdp", "log_population", "log_energy_per_capita",
     "industry_share_gdp", "fossil_pct", "natural_resource_rents_per_gdp",
     "implementation_capacity_z", "political_stability_z", "democratic_legitimacy_z"]
    + event_dummies
)

df_es[cols_for_na_check].isna().sum().sum()
# should be 0


0

In [46]:
import statsmodels.formula.api as smf

event_term = " + ".join([f'Q("{col}")' for col in event_dummies])

formula_es = f"""
co2_per_capita_3yr_change ~
    {event_term}
    + log_gdp
    + log_population
    + log_energy_per_capita
    + industry_share_gdp
    + fossil_pct
    + natural_resource_rents_per_gdp
    + implementation_capacity_z
    + political_stability_z
    + democratic_legitimacy_z
    + C(country) + C(year)
"""

es_model = smf.ols(formula=formula_es, data=df_es).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_es["country"]}
)

print(es_model.summary())


                                OLS Regression Results                               
Dep. Variable:     co2_per_capita_3yr_change   R-squared:                       0.221
Model:                                   OLS   Adj. R-squared:                  0.176
Method:                        Least Squares   F-statistic:                     8.855
Date:                       Sat, 20 Dec 2025   Prob (F-statistic):           2.84e-26
Time:                               19:54:20   Log-Likelihood:                -5173.7
No. Observations:                       3830   AIC:                         1.077e+04
Df Residuals:                           3620   BIC:                         1.208e+04
Df Model:                                209                                         
Covariance Type:                     cluster                                         
                                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

C:\Users\willn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1888: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 209, but rank is 48
  warnings.warn('covariance of constraints does not have full '


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Extract coefficients and 95% CIs for event dummies
K = 8
baseline = -1
event_times = [k for k in range(-K, K + 1) if k != baseline]

coefs = []
ci_low = []
ci_high = []

ci = model_es.conf_int()
params = model_es.params

for k in event_times:
    label = f'rel_{k}'
    if label in params.index:
        coefs.append(params[label])
        ci_low.append(ci.loc[label, 0])
        ci_high.append(ci.loc[label, 1])
    else:
        coefs.append(np.nan)
        ci_low.append(np.nan)
        ci_high.append(np.nan)

# Insert baseline (0, 0, 0) at position for t=-1
all_times = list(range(-K, K + 1))
baseline_pos = all_times.index(baseline)

coefs_full = coefs[:baseline_pos] + [0] + coefs[baseline_pos:]
ci_low_full = ci_low[:baseline_pos] + [0] + ci_low[baseline_pos:]
ci_high_full = ci_high[:baseline_pos] + [0] + ci_high[baseline_pos:]

print("Pre-treatment coefficients (should be near zero for parallel trends):")
pre = [(t, c, lo, hi) for t, c, lo, hi in zip(all_times, coefs_full, ci_low_full, ci_high_full) if t < 0]
for t, c, lo, hi in pre:
    sig = "***" if (lo > 0 or hi < 0) else "   "
    print(f"  t={t:3d}: coef={c:7.4f}  CI=[{lo:.4f}, {hi:.4f}] {sig}")

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 6))

err_low  = [c - lo for c, lo in zip(coefs_full, ci_low_full)]
err_high = [hi - c for c, hi in zip(coefs_full, ci_high_full)]

ax.errorbar(
    all_times, coefs_full,
    yerr=[err_low, err_high],
    fmt='o', color='steelblue', ecolor='steelblue',
    capsize=4, linewidth=1.5, markersize=5, label='Point estimate + 95% CI'
)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(-0.5, color='red', linewidth=1.2, linestyle='--', label='Treatment onset')
ax.axvspan(-K - 0.5, -0.5, alpha=0.05, color='grey', label='Pre-treatment window')

ax.set_xlabel('Years Relative to Carbon Tax Adoption', fontsize=12)
ax.set_ylabel('Effect on CO₂/Capita 3-Year Trend', fontsize=12)
ax.set_title('Event Study: Carbon Tax Effect Over Time\n(Baseline = Year Before Adoption)', fontsize=13)
ax.set_xticks(all_times)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/event_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/event_study.png")

## Event Study Interpretation

**Parallel Trends Assessment:**
- Pre-treatment coefficients (t = -8 to t = -2): inspect printed output above — any `***` rows indicate a violation
- Conclusion: fill in after running

**Dynamic Treatment Effects:**
- Immediate effect (t=0): fill in after running
- Pattern: fill in after running — consistent with ~1%/year decay finding?

**Implication for DiD validity:** fill in after running